In [1]:
import pandas as pd
import numpy as np

# 1. Load the data
mapping_df = pd.read_csv('Summary Evaluation Mapping.csv')
survey_df = pd.read_csv('Summary_Evaluation_Study_-_Best_and_Worst.csv')

# 2. Identify the survey response columns
# Usually: 0: Timestamp, 1: Username, 2: Name. Responses start from index 3.
# Each question has two columns: BEST and WORST.
response_cols = survey_df.columns[3:]
num_questions = len(response_cols) // 2
num_participants = len(survey_df)
total_evaluations = num_participants * num_questions

# 3. Initialize counts for each model
models = ['Original', 'Hierarchical', 'NLI', 'Fact']
# We track Best, Worst, and Neutral (Middle Ground)
metrics = {model: {'Best': 0, 'Worst': 0, 'Neutral': 0} for model in models}

# 4. Map options to models and calculate counts
for _, row in survey_df.iterrows():
    for q_idx in range(num_questions):
        q_num = q_idx + 1
        # Extract the letters chosen by the participant
        best_opt = str(row[response_cols[2 * q_idx]]).strip().upper()
        worst_opt = str(row[response_cols[2 * q_idx + 1]]).strip().upper()
        
        # Get the mapping for this specific question
        q_map = mapping_df[mapping_df['Question Number'] == q_num].iloc[0]
        
        # Identify which model corresponds to each option A, B, C, D
        option_to_model = {
            'A': q_map['Option A Model'],
            'B': q_map['Option B Model'],
            'C': q_map['Option C Model'],
            'D': q_map['Option D Model']
        }
        
        # Record Best and Worst
        best_model = option_to_model.get(best_opt)
        worst_model = option_to_model.get(worst_opt)
        
        if best_model:
            metrics[best_model]['Best'] += 1
        if worst_model:
            metrics[worst_model]['Worst'] += 1
            
        # The other two models are "Neutral"
        for opt_letter, model_name in option_to_model.items():
            if opt_letter != best_opt and opt_letter != worst_opt:
                metrics[model_name]['Neutral'] += 1

# 5. Compile and calculate advanced metrics
results = pd.DataFrame.from_dict(metrics, orient='index')
results['Net Score'] = results['Best'] - results['Worst']
results['Win Rate (%)'] = (results['Best'] / total_evaluations * 100).round(2)
results['Loss Rate (%)'] = (results['Worst'] / total_evaluations * 100).round(2)
results['Neutral Rate (%)'] = (results['Neutral'] / (total_evaluations * 2) * 100).round(2) # *2 because 2 neutrals per Q

# Sort by Best performance
results = results.sort_values(by='Best', ascending=False)

# 6. Display results
print(f"Total Participants: {num_participants}")
print(f"Total Questions per Participant: {num_questions}")
print(f"Total Evaluations per Model: {total_evaluations}\n")
print("Summary Evaluation Results:")
print(results)

# Optional: Save results to a CSV
results.to_csv('survey_metrics_summary.csv')

Total Participants: 3
Total Questions per Participant: 20
Total Evaluations per Model: 60

Summary Evaluation Results:
              Best  Worst  Neutral  Net Score  Win Rate (%)  Loss Rate (%)  \
Original        28      5       27         23         46.67           8.33   
Hierarchical    16      8       36          8         26.67          13.33   
NLI             12     17       31         -5         20.00          28.33   
Fact             4     30       26        -26          6.67          50.00   

              Neutral Rate (%)  
Original                 22.50  
Hierarchical             30.00  
NLI                      25.83  
Fact                     21.67  


In [2]:
import pandas as pd
import numpy as np

# load data
df = pd.read_csv("survey_metrics_summary.csv")

# total responses
df["Total"] = df["Best"] + df["Worst"] + df["Neutral"]

# Net Preference Score
df["Net Preference Score"] = (df["Best"] - df["Worst"]) / df["Total"]

# Preference share
df["Preference Share"] = df["Best"] / df["Total"]

# Rejection rate
df["Rejection Rate"] = df["Worst"] / df["Total"]

# Preference ratio
df["Preference Ratio"] = df["Best"] / (df["Worst"] + 1e-6)

# Relative ranking score (MaxDiff metric)
df["Relative Score"] = (df["Best"] - df["Worst"]) / (df["Best"] + df["Worst"])

# Sort by best model
df = df.sort_values("Net Preference Score", ascending=False)

print(df)

# save results
df.to_csv("summary_model_metrics.csv", index=False)

     Unnamed: 0  Best  Worst  Neutral  Net Score  Win Rate (%)  Loss Rate (%)  \
0      Original    28      5       27         23         46.67           8.33   
1  Hierarchical    16      8       36          8         26.67          13.33   
2           NLI    12     17       31         -5         20.00          28.33   
3          Fact     4     30       26        -26          6.67          50.00   

   Neutral Rate (%)  Total  Net Preference Score  Preference Share  \
0             22.50     60              0.383333          0.466667   
1             30.00     60              0.133333          0.266667   
2             25.83     60             -0.083333          0.200000   
3             21.67     60             -0.433333          0.066667   

   Rejection Rate  Preference Ratio  Relative Score  
0        0.083333          5.599999        0.696970  
1        0.133333          2.000000        0.333333  
2        0.283333          0.705882       -0.172414  
3        0.500000          